In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q transformers peft trl datasets accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 40.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.4 MB/s eta 0:00:00


In [ ]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.9 MB/s eta 0:00:00


In [ ]:
!python /content/drive/MyDrive/Finetuning/train_qwen.py \
    --train_file /content/drive/MyDrive/Finetuning/datachat/train.jsonl \
    --val_file /content/drive/MyDrive/Finetuning/datachat/val.jsonl  \
    --output_dir /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading tokenizer/model: Qwen/Qwen2.5-1.5B-Instruct
config.json: 100% 660/660 [00:00<00:00, 3.00MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 63.6MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 115MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 154MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 8.26M/3.09G [00:01<06:17, 8.16MB/s]
model.safetensors: downloading bytes:   1% 44.9M/3.09G [00:01<00:57, 52.5MB/s, 2.62MB/s  ]
model.safetensors: download

In [ ]:
!python /content/drive/MyDrive/Finetuning/generate.py \
    --adapter_dir /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final \
    --base_model Qwen/Qwen2.5-1.5B-Instruct

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
Loading weights: 100% 338/338 [00:00<00:00, 1287.42it/s]
Loading LoRA adapter from: /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final

Generating on device: cuda


[Belethor] — Greeting a customer entering his shop
----------------------------------------------------------------------
  (1) e hear there were some troubles with Throat of Nords recently... I suppose we're not to blame for that either then!
  (2) t wines... now what will it be?

[Nazeem] — Boasting about the Cloud District
--------------------------------

In [ ]:
!python /content/drive/MyDrive/Finetuning/generate.py \
    --adapter_dir /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final \
    --base_model Qwen/Qwen2.5-1.5B-Instruct

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
Loading weights: 100% 338/338 [00:00<00:00, 3272.49it/s]
Loading LoRA adapter from: /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final

Generating on device: cuda


[Belethor] — Greeting a customer entering his shop
----------------------------------------------------------------------
  (1) I see you're interested... You'd like to know more about my work? I'm always happy to share with customers! Come into your own little corner of this world while it lasts. It's
  (2) I see you've made it through another cold season..

In [ ]:
count = 0
for line in open("/content/drive/MyDrive/Finetuning/datachat/train.jsonl", encoding="utf-8"):
    if "[[" in line or "]]" in line:
        count += 1
print(f"{count} examples with leaked wiki markup")

0 examples with leaked wiki markup


In [ ]:
import torch
import json
import math
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

def compute_perplexity(model, tokenizer, jsonl_path, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with open(jsonl_path, encoding="utf-8") as f:
        examples = [json.loads(line) for line in f]

    for ex in examples:
        # Adjust this to match your actual test.jsonl schema (chat vs. text format)
        text = ex.get("text") or tokenizer.apply_chat_template(ex["messages"], tokenize=False)
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            # loss is mean per-token cross-entropy; multiply by token count to get sum
            num_tokens = inputs["input_ids"].shape[1]
            total_loss += outputs.loss.item() * num_tokens
            total_tokens += num_tokens

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    return perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
test_path = "/content/drive/MyDrive/Finetuning/datachat/test.jsonl"  # adjust to your actual path
adapter_dir = "/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final"

tokenizer = AutoTokenizer.from_pretrained(adapter_dir)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct").to(device)
base_ppl = compute_perplexity(base_model, tokenizer, test_path, device)
print(f"Base model perplexity: {base_ppl:.3f}")

print("Loading fine-tuned model...")
finetuned_model = PeftModel.from_pretrained(base_model, adapter_dir).to(device)
finetuned_ppl = compute_perplexity(finetuned_model, tokenizer, test_path, device)
print(f"Fine-tuned model perplexity: {finetuned_ppl:.3f}")

print(f"\nImprovement: {base_ppl - finetuned_ppl:.3f} ({(1 - finetuned_ppl/base_ppl) * 100:.1f}% lower)")

Loading base model...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model perplexity: 61.383
Loading fine-tuned model...
Fine-tuned model perplexity: 2.444

Improvement: 58.938 (96.0% lower)


In [ ]:
!python /content/drive/MyDrive/Finetuning/merge_lora.py \
    --adapter_dir /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final \
    --output_dir /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-merged

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 338/338 [00:07<00:00, 47.52it/s]
Loading tokenizer from adapter dir (has the chat template): /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final
Loading LoRA adapter from: /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-lora/final
Merging adapter into base weights...
Saving merged model to: /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-merged
Writing model shards: 100% 1/1 [00:21<00:00, 21.23s/it]

Done. This fold

In [ ]:
!git clone https://github.com/ggerganov/llama.cpp

Cloning into 'llama.cpp'...
remote: Enumerating objects: 106574, done.
remote: Counting objects: 100% (154/154), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 106574 (delta 102), reused 70 (delta 70), pack-reused 106420 (from 3)
Receiving objects: 100% (106574/106574), 413.41 MiB | 17.32 MiB/s, done.
Resolving deltas: 100% (74617/74617), done.


In [ ]:
!pip install -r llama.cpp/requirements.txt


Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 7.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 75.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 108.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q transformers==4.46.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 123.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 110.0 MB/s eta 0:00:00


In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

!python llama.cpp/convert_hf_to_gguf.py \
    /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-merged \
    --outfile /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf \
    --outtype f16

INFO:hf-to-gguf:Loading model: qwen-skyrim-merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.floa

In [ ]:
%cd /content/llama.cpp
!cmake --build build --config Release -j 2 --target llama-quantize

/content/llama.cpp
[  0%] Built target cpp-httplib
[  4%] Built target ggml-base
[  4%] Built target llama-common-base
[ 13%] Built target ggml-cpu
[ 13%] Built target ggml
[ 84%] Built target llama
[100%] Built target llama-common
[100%] Building CXX object tools/quantize/CMakeFiles/llama-quantize-impl.dir/quantize.cpp.o
[100%] Linking CXX shared library ../../bin/libllama-quantize-impl.so
[100%] Built target llama-quantize-impl
[100%] Building CXX object tools/quantize/CMakeFiles/llama-quantize.dir/main.cpp.o
[100%] Linking CXX executable ../../bin/llama-quantize
[100%] Built target llama-quantize


In [ ]:
!cmake --build build --config Release -j 2 --target llama-perplexity

!/content/llama.cpp/build/bin/llama-perplexity \
    -m /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf \
    -f /content/drive/MyDrive/Finetuning/datachat/test.jsonl

[  0%] Built target cpp-httplib
[  4%] Built target ggml-base
[  4%] Built target llama-common-base
[ 12%] Built target ggml-cpu
[ 12%] Built target ggml
[ 82%] Built target llama
[ 97%] Built target llama-common
[ 97%] Built target llama-perplexity-impl
[100%] Built target llama-perplexity
0.00.007.637 E gguf_init_from_file: failed to open GGUF file '/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf' (No such file or directory)
0.00.007.718 E llama_model_load: error loading model: llama_model_loader: failed to load model from /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf
0.00.007.730 E llama_model_load_from_file_impl: failed to load model
0.00.007.767 E common_fit_params: encountered an error while trying to fit params to free device memory: failed to load model
0.00.007.917 E gguf_init_from_file: failed to open GGUF file '/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf' (No such file or directory)
0.00.007.939 E llama_

In [ ]:
!ls -la /content/drive/MyDrive/Finetuning/checkpoints/


total 3021173
drwx------ 2 root root       4096 Jul 23 05:40 gpt2-skyrim-lora
-rw------- 1 root root 3093668832 Jul 31 12:24 qwen-skyrim.gguf
drwx------ 5 root root       4096 Jul 29 11:25 qwen-skyrim-lora
drwx------ 2 root root       4096 Jul 31 11:57 qwen-skyrim-merged


In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
    /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf \
    /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf \
    Q4_K_M

llama_print_build_info: build = 10210 (000547513)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf' to '/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                   

In [ ]:
!/content/llama.cpp/build/bin/llama-quantize \
    /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf \
    /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q5_K_M.gguf \
    Q5_K_M

llama_print_build_info: build = 10210 (000547513)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf' to '/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q5_K_M.gguf' as Q5_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                   

In [ ]:
%cd /content/llama.cpp
!/content/llama.cpp/build/bin/llama-perplexity \
    -m /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q4_K_M.gguf \
    -f /content/drive/MyDrive/Finetuning/datachat/test.jsonl

/content/llama.cpp
0.00.639.640 W load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
0.05.694.891 I 
0.05.695.062 I system_info: n_threads = 1 (n_threads_batch = 1) / 2 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
0.05.695.105 I perplexity: tokenizing the input ..
0.05.746.234 I perplexity: tokenization took 51.125 ms
0.05.746.383 I perplexity: calculating perplexity over 52 chunks, n_ctx=512, batch_size=2048, n_seq=4
1.40.669.561 I perplexity: 94.92 seconds per pass - ETA 20.57 minutes
[1]1.9228,[2]2.0540,[3]2.1359,[4]2.1774,[5]2.1307,[6]2.1215,[7]2.0871,[8]2.1650,[9]2.0936,[10]2.0513,[11]2.0241,[12]2.0709,[13]2.0706,[14]2.0860,[15]2.0645,[16]2.0345,[17]2.0456,[18]2.0538,[19]2.0819,[20]2.0993,[21]2.0892,[22]2.0630,[23]2.0692,[24]2.0541,[25]2.0517,[26]2.0714,[27]2.0716,[28]2.0722,[29]2.0899,[30]2.1075,[31]2.0976,[3

In [ ]:
%cd /content/llama.cpp
!/content/llama.cpp/build/bin/llama-perplexity \
    -m /content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-Q5_K_M.gguf \
    -f /content/drive/MyDrive/Finetuning/datachat/test.jsonl

/content/llama.cpp
0.00.638.386 W load: control-looking token: 128247 '</s>' was not control-type; this is probably a bug in the model. its type will be overridden
0.03.123.005 I 
0.03.123.138 I system_info: n_threads = 1 (n_threads_batch = 1) / 2 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | AVX512 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 
0.03.123.159 I perplexity: tokenizing the input ..
0.03.184.454 I perplexity: tokenization took 61.287 ms
0.03.184.606 I perplexity: calculating perplexity over 52 chunks, n_ctx=512, batch_size=2048, n_seq=4
4.05.348.887 I perplexity: 242.16 seconds per pass - ETA 52.47 minutes
[1]1.9163,[2]2.0327,[3]2.1287,[4]2.1837,[5]2.1261,[6]2.1164,[7]2.0900,[8]2.1632,[9]2.0916,[10]2.0465,[11]2.0251,[12]2.0745,[13]2.0740,[14]2.0929,[15]2.0745,[16]2.0439,[17]2.0548,[18]2.0602,[19]2.0924,[20]2.1090,[21]2.1023,[22]2.0764,[23]2.0838,[24]2.0694,[25]2.0660,[26]2.0865,[27]2.0880,[28]2.0886,[29]2.1036,[30]2.1223,[31]2.1117,[

In [ ]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

In [7]:
from huggingface_hub import HfApi, create_repo

repo_id = "AnuBht/qwen-skyrim-merged"

create_repo(repo_id, private=True, exist_ok=True)  # exist_ok avoids error if it's already there

api = HfApi()
api.upload_folder(
    folder_path="/content/drive/MyDrive/Finetuning/checkpoints/qwen-skyrim-merged",
    repo_id=repo_id,
)

CommitInfo(commit_url='https://huggingface.co/AnuBht/qwen-skyrim-merged/commit/7dee9bd535894d4da965cd5b8e275f69cfce335d', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7dee9bd535894d4da965cd5b8e275f69cfce335d', pr_url=None, repo_url=RepoUrl('https://huggingface.co/AnuBht/qwen-skyrim-merged', endpoint='https://huggingface.co', repo_type='model', repo_id='AnuBht/qwen-skyrim-merged'), pr_revision=None, pr_num=None)